# Speculative Decoding

**Prerequisites:** Python basics and the relevant model-serving concepts.  **Estimated time:** 35 minutes.

## Objectives

- Account for draft acceptance and the limits imposed by draft and verification cost.
- Explain at least one failure boundary and a diagnostic.
- Run deterministic assertions against a small synthetic example.


## Model and assumptions

Sweep acceptance prefixes and compare symbolic work, not wall-clock speed. All quantities below are deliberately small, deterministic teaching values. The paired local implementations are dependency-light (the MoE example uses NumPy) and do not call a model or network service. A synthetic proxy establishes accounting behavior, not production quality or speed.


In [ ]:
# Load the paired standalone implementation from the repository root.
import runpy
impl = runpy.run_path('modern-ai/implementations/28-speculative-decoding.py')


## Transparent baseline

Inspect the short implementation above: the critical state and accounting are visible, so you can change one assumption at a time. In a real system, replace the toy inputs with representative traces and validate quality as well as resource use.


In [ ]:
rows=[impl['account'](4,a,.15,.8,1.0) for a in range(5)]
print([(r['acceptance_rate'],r['speedup_bound']) for r in rows])
assert rows[0]['accepted']==0 and rows[-1]['accepted']==4


## Comparison and metric

Use the printed counts, mask density, error, latency distribution, or decision labels to compare the alternatives. The selected metric is a proxy for a specific resource or behavior; it is not a substitute for end-to-end evaluation.

In [ ]:
# Edge case: reject invalid inputs or expose degraded behavior instead of hiding it.
try:
    impl['account'](4, 5, .1, .5, 1)
except (ValueError, MemoryError, IndexError) as exc:
    print(type(exc).__name__, exc)


## Failure cases and exercise

1. Change the topic parameter that exercises a boundary. Predict which metric changes before running the cell.
2. Identify a case where the proxy gives a misleading decision. State what real trace or quality label you would collect before deployment.

**Exercise:** Use the invariant below as a starting test, then add a second case that probes a boundary relevant to your workload.

In [ ]:
poor=impl['account'](4,0,.15,.8,1.0); good=impl['account'](4,4,.15,.8,1.0)
assert good['acceptance_rate'] > poor['acceptance_rate']


## Summary and next lessons

This notebook made one speculative decoding mechanism inspectable with deterministic accounting. Revisit the concept note at `../concepts/28-speculative-decoding.md`, then compare with the neighboring serving lessons and evaluate on a representative workload before making a system choice.
